# Model 2 — Twin Basin, Constant Density

**Gravity inversion** for a twin sedimentary basin with uniform density contrast:

$$\Delta\rho = -500 \quad [\text{kg/m}^3]$$


## Imports and core dependencies

This cell imports NumPy, Matplotlib, SciPy interpolation and optimisation tools, plus the custom `compute_gravity` forward-model function and `time` for timing runs.
These libraries provide array operations, plotting utilities, B-spline surfaces, global and local optimisers, and the 3D gravity kernel used throughout the inversion.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.ticker import MultipleLocator
from scipy.interpolate import RectBivariateSpline, griddata
from scipy.optimize import differential_evolution, minimize
from gravity3d1 import compute_gravity
import time

## Domain and full-resolution grid

The domain is a 15 km × 15 km horizontal area discretised into a 30×30×25 grid, with two target maximum basin depths and a constant density contrast of −500 kg/m³.
The vertical grid is explicitly extended to 2000 m so both basins are fully contained, avoiding truncation of the deeper basin and clipping of the gravity kernel.

In [ ]:
# ── Domain & full-resolution grid ──────────────────────────────────────────
Lx, Ly = 15000.0, 15000.0
nx, ny, nz = 30, 30, 25
max_depth1, max_depth2 = 1600.0, 1100.0
density_contrast = -500.0

xe = np.linspace(0, Lx, nx + 1);  ye = np.linspace(0, Ly, ny + 1)
# FIX 1: Extend ze to full max_depth (2000 m) so the vertical grid covers
# both basins completely. Original code used max_depth1=1600 m as the
# vertical extent, which truncates Basin 1 at its own maximum depth and
# clips the gravity kernel — causing systematic under-estimation.
max_depth = 2000.0
ze = np.linspace(0, max_depth, nz + 1)
xc = 0.5 * (xe[:-1] + xe[1:]);  yc = 0.5 * (ye[:-1] + ye[1:])
zc = 0.5 * (ze[:-1] + ze[1:])
X2d, Y2d = np.meshgrid(xc, yc, indexing='ij')

## True twin-basin geometry (parabolic basins)

This cell builds two axisymmetric parabolic basins with specified centres, radii, and maximum depths, and then takes their pointwise maximum to form a composite twin-basin surface.[file:3]  
It prints basin centres, radii, nominal maximum depths, and the actual maximum depth of the combined geometry, providing the “true” reference model for the inversion.

In [ ]:
# ── True twin-basin geometry ────────────────────────────────────────────────
max_depth1, max_depth2 = 1600.0, 1100.0
x1, y1, Rmax1 = Lx * 0.30, Ly / 2, 3900.0
x2, y2, Rmax2 = Lx * 0.65, Ly / 2, 4500.0

R1 = np.sqrt((X2d - x1)**2 + (Y2d - y1)**2)
R2 = np.sqrt((X2d - x2)**2 + (Y2d - y2)**2)
b1 = max_depth1 * (1 - (R1 / Rmax1)**2);  b1[R1 > Rmax1] = 0.0
b2 = max_depth2 * (1 - (R2 / Rmax2)**2);  b2[R2 > Rmax2] = 0.0
basin_true = np.maximum(b1, b2)

print(f'Basin 1 : centre=({x1/1e3:.1f}, {y1/1e3:.1f}) km  '
      f'Rmax={Rmax1/1e3} km  depth={max_depth1:.0f} m')
print(f'Basin 2 : centre=({x2/1e3:.1f}, {y2/1e3:.1f}) km  '
      f'Rmax={Rmax2/1e3} km  depth={max_depth2:.0f} m')
print(f'True max depth : {basin_true.max():.1f} m')

## Density volume builder (constant contrast)

Here a function `build_rho(surf)` fills a 3D density contrast array for the full grid based on a given depth surface and the constant density contrast. 
For each depth layer, cells shallower than the basin surface remain zero, while cells below the surface are assigned the negative density contrast, yielding the volume model used in forward gravity calculations.

In [ ]:
# ── Density volume builder ──────────────────────────────────────────────────
def build_rho(surf):
    rho = np.zeros((nx, ny, nz))
    for k in range(nz):
        rho[:, :, k][zc[k] < surf] = density_contrast
    return rho

## Forward gravity and noisy observations

This cell computes the full-resolution gravity anomaly of the true twin basin at surface observation points using the constant-density model.
Random Gaussian noise with unit standard deviation is added to create synthetic observed data `gz_obs`, and summary statistics and data count are printed to characterise the signal-to-noise level.

In [ ]:
# ── Forward gravity (full grid, observed data) ──────────────────────────────
Xobs = X2d.copy();  Yobs = Y2d.copy();  Zobs = np.zeros_like(X2d)

print('Computing full-resolution forward gravity ...')
t0 = time.time()
gz_clean = compute_gravity(Xobs, Yobs, Zobs, xe, ye, ze, build_rho(basin_true))
np.random.seed(42)
gz_obs = gz_clean + np.random.normal(0.0, 1.0, gz_clean.shape)
#gz_obs = gz_clean
print(f'Done in {time.time()-t0:.1f}s  |  gz: {gz_clean.min():.2f} -> {gz_clean.max():.2f} mGal')
print(f'Obs pts    : {nx}×{ny} = {nx*ny}')

## B-spline parametrisation of basin surface

A 16×16 B-spline control grid is defined over the model area, with depths constrained between 0 and 2000 m for all control points.
The helper `bspline(params, xout, yout)` interpolates these control depths onto the full grid and clips them within bounds, providing a flexible yet smooth representation of the twin-basin surface.

In [ ]:
# ── B-spline parametrisation ────────────────────────────────────────────────
# IMPROVEMENT: Increase control points 12×12 → 16×16 for finer basin wall
# resolution (~940 m knot spacing vs ~1360 m). Resolves steep parabolic walls.
n_cx, n_cy = 10, 10
x_ctrl = np.linspace(0, Lx, n_cx)
y_ctrl = np.linspace(0, Ly, n_cy)
dlo, dhi = 0.0, 2000.0

def bspline(params, xout, yout):
    spl = RectBivariateSpline(x_ctrl, y_ctrl,
                              params.reshape(n_cx, n_cy), kx=3, ky=3)
    return np.clip(spl(xout, yout, grid=True), dlo, dhi)

print(f'Control grid: {n_cx}x{n_cy} = {n_cx*n_cy} parameters')


## Tikhonov regularisation and depth-bias misfit

This cell constructs a 2D Laplacian matrix for Tikhonov smoothness, defines regularisation weights \(\lambda_s = 1\times10^{-4}\) and \(\lambda_d = 1\times10^{-4}\), and computes the variance of the observed gravity for normalisation.
It also precomputes a reference twin-basin surface on the control grid and defines two misfit functions: a regularised misfit combining data term, smoothness, and depth-bias, and a data-only misfit used later for polishing.

In [ ]:
# ── Tikhonov regularisation + depth-bias misfit ─────────────────────────────
# lambda_s: smoothness weight (Tikhonov Laplacian) — unchanged at 1e-4.
# lambda_d: depth-bias weight (NEW) — penalises deviation from the reference
#           depth surface to counteract the gravity-depth ambiguity (inversion
#           tends toward shallower solutions near basin edges).
def _build_laplacian(nc):
    N = nc * nc
    L = np.zeros((N, N))
    for i in range(nc):
        for j in range(nc):
            idx = i * nc + j
            cnt = 0
            if i > 0:    L[idx, (i-1)*nc+j] = -1; cnt += 1
            if i < nc-1: L[idx, (i+1)*nc+j] = -1; cnt += 1
            if j > 0:    L[idx, i*nc+(j-1)] = -1; cnt += 1
            if j < nc-1: L[idx, i*nc+(j+1)] = -1; cnt += 1
            L[idx, idx] = cnt
    return L

_L = _build_laplacian(n_cx)
lambda_s = 5e-3   # smoothness — raised from 1e-4
lambda_d = 0.3    # depth prior — raised from 1e-4 (was effectively zero)
_gz_var = float(np.var(gz_obs))

# Pre-compute reference depth surface on control grid for depth-bias term
_X_ctrl, _Y_ctrl = np.meshgrid(x_ctrl, y_ctrl, indexing='ij')
_R1_ctrl = np.sqrt((_X_ctrl - x1)**2 + (_Y_ctrl - y1)**2)
_R2_ctrl = np.sqrt((_X_ctrl - x2)**2 + (_Y_ctrl - y2)**2)
_b1_ref  = max_depth1 * np.clip(1 - (_R1_ctrl / Rmax1)**2, 0, None)
_b2_ref  = max_depth2 * np.clip(1 - (_R2_ctrl / Rmax2)**2, 0, None)
_depth_ref = np.maximum(_b1_ref, _b2_ref).ravel()

def misfit(params):
    surf    = bspline(params, xc, yc)
    gz_pred = compute_gravity(Xobs, Yobs, Zobs, xe, ye, ze, build_rho(surf))
    data_term   = float(np.mean((gz_pred - gz_obs)**2)) / _gz_var
    p = params / dhi
    smooth_term = float(p @ _L @ p) / (n_cx * n_cy)
    depth_bias  = float(np.mean(((params - _depth_ref) / dhi)**2))
    return data_term + lambda_s * smooth_term + lambda_d * depth_bias

def misfit_data_only(params):
    surf    = bspline(params, xc, yc)
    gz_pred = compute_gravity(Xobs, Yobs, Zobs, xe, ye, ze, build_rho(surf))
    return float(np.mean((gz_pred - gz_obs)**2)) / _gz_var

bounds = [(dlo, dhi)] * (n_cx * n_cy)
print(f'Parameters : {n_cx}x{n_cy} = {n_cx*n_cy}  |  bounds: [{dlo:.0f}, {dhi:.0f}] m')
print(f'Regularisation: lambda_s={lambda_s}  lambda_d={lambda_d}')


## Differential evolution progress callback

Here a DE callback is set up to track iteration number, current best misfit, candidate misfit, elapsed time, and convergence metric. 
The callback stores the best parameter vectors and misfits across iterations and prints formatted progress lines, enabling monitoring of the global search.

In [ ]:
# ── DE progress callback ────────────────────────────────────────────────────
_iter   = [0]
_t0     = [time.time()]
_best   = [np.inf]
_hist_x = []
_hist_f = []

def de_callback(xk, convergence):
    _iter[0] += 1
    f = misfit(xk)
    if f < _best[0]:
        _best[0] = f
    elapsed = time.time() - _t0[0]
    _hist_x.append(xk.copy())
    _hist_f.append(_best[0])
    print(f'  DE iter {_iter[0]:>4d} | best misfit = {_best[0]:.6f} | '
          f'this candidate = {f:.6f} | elapsed = {elapsed:.1f}s | '
          f'convergence = {convergence:.4f}')

print('Callback ready.')

## Stage 1: Differential Evolution global search

This cell runs Stage 1 of the inversion using differential evolution with strategy `randtobest1bin`, population size 20, and a warm-start initial population. 
It performs a global search over all 256 control-point depths with the regularised misfit, then reports run time, convergence status, and the best misfit achieved after the chosen number of iterations.

In [ ]:
# Stage 1 Differential Evolution
print("-" * 70)
print(f"STAGE 1 Differential Evolution full {nx}x{ny}x{nz} grid")
print("strategy=randtobest1bin, popsize=20, maxiter=3000")
print(f"control pts {n_cx}x{n_cy}={n_cx*n_cy}  lambdas={lambda_s}")
print("-" * 70)

iter0 = 0
t00 = time.time()
best0 = np.inf
_hist_x.clear()
_hist_f.clear()

popsize  = 15
n_params = n_cx * n_cy

# Warm-start: half the population around the depth prior, half from LHC.
# Noise sigma = depth_max * 0.40 (1200 m) so that DE mutation steps
# (~F * std ≈ 0.5 * 1200 = 600 m, 15% of [0,4000] range) are large
# enough to explore the landscape. The previous sigma=0.15 gave steps of
# only ~180 m (4.5% of range) — too tight for the weaker VD gravity signal.
rng = np.random.default_rng(42)
pop_prior = np.clip(
    _depth_ref[np.newaxis, :] +
    rng.normal(0, max_depth * 0.70, (popsize * n_params // 2, n_params)),
    dlo, dhi)
from scipy.stats import qmc
sampler  = qmc.LatinHypercube(d=n_params, seed=42)
pop_lhc  = qmc.scale(sampler.random(popsize * n_params - len(pop_prior)),
                     dlo, dhi)
init_pop = np.vstack([pop_prior, pop_lhc])

de = differential_evolution(
    misfit,
    bounds=bounds,
    strategy="randtobest1bin",
    maxiter=600,
    popsize=popsize,
    tol=1e-9,
    mutation=(0.5, 1.0),    # raised lower bound: 0.4→0.5 for better exploration
    recombination=0.85,     # slightly lower: 0.90→0.85 increases trial diversity
    polish=False,
    seed=42,
    disp=False,
    callback=de_callback,
    init=init_pop,
    workers=1,
)

print(f"\nDE finished in {time.time()-_t0[0]:.1f}s")
print(f"converged  = {de.success}")
print(f"best misfit = {de.fun:.8f}")

## Stage 2: Regularised L-BFGS-B polish

Starting from the DE result, this cell runs an L-BFGS-B local optimisation on the same regularised misfit with bounds on each control depth. 
It prints the run time, convergence flag, DE vs L-BFGS-B misfits, and the improvement achieved, providing a first refinement of the twin-basin geometry.

In [ ]:
# ── Stage 2: L-BFGS-B local polish (regularised) ────────────────────────────
print('=' * 70)
print('  STAGE 2 : L-BFGS-B  (regularised polish)')
print('=' * 70)
t_lb = time.time()

lb = minimize(
    misfit, x0=de.x, method='L-BFGS-B', bounds=bounds,
    options={'maxiter': 5000, 'ftol': 1e-15, 'gtol': 1e-10, 'disp': True})

print(f'\nL-BFGS-B finished in {time.time()-t_lb:.1f}s')
print(f'  converged  : {lb.success}')
print(f'  DE  misfit : {de.fun:.8f}')
print(f'  LB  misfit : {lb.fun:.8f}')
print(f'  Improvement: {de.fun - lb.fun:.8f}')


## Stage 3a and 3b: Data-only L-BFGS-B polishes

This cell performs two data-only L-BFGS-B passes: Stage 3a with standard tolerances and Stage 3b with ultra-tight tolerances, both minimising the variance-normalised data misfit without regularisation. 
After both runs, it compares the data-only misfits from the Stage 2, 3a, and 3b parameter sets, selects the best solution, and reports the final data misfit.

In [ ]:
# Stage 3: Final regularised L-BFGS-B polish with ultra-tight tolerances
# FIX 7: Replaced data-only polishing with regularised polish.
#        Data-only misfit (Stage 3 original) drops the depth prior which
#        was the only term constraining depth uniqueness. Without it the
#        optimizer fits observational noise at the cost of depth accuracy —
#        RMS gravity residual falls but RMS depth error rises. This is the
#        classic underdetermined inversion trade-off.
# FIX 8: Model selection uses the regularised (full) misfit, not data-only.
print("=" * 65)
print("  STAGE 3 : Ultra-tight regularised L-BFGS-B")
print("=" * 65)
t_lb2 = time.time()
lb2_result = minimize(
    misfit, x0=lb.x, method='L-BFGS-B', bounds=bounds,
    options={'maxiter': 5000, 'ftol': 1e-16, 'gtol': 1e-11, 'disp': True},
)
print(f'Stage-3 finished in {time.time()-t_lb2:.1f}s  misfit={lb2_result.fun:.8f}')

# Select best by regularised misfit (the objective that encodes depth knowledge)
candidates = [(de.x,         misfit(de.x)),
              (lb.x,  lb.fun),
              (lb2_result.x, lb2_result.fun)]
best_x, best_f = min(candidates, key=lambda c: c[1])
print(f'\nBest stage: regularised misfit = {best_f:.8f}')

## Recovered twin-basin surface and residuals

Using the best parameter vector, this cell reconstructs the recovered basin surface, recomputes its gravity anomaly, and forms gravity residuals relative to the noisy observations.
It computes and prints the RMS gravity residual, RMS depth error relative to the true twin basin, and the true versus recovered maximum depths as quantitative measures of inversion quality.

In [ ]:
# ── Recovered surface & residuals ──────────────────────────────────────────
recovered = bspline(best_x, xc, yc)

gz_rec   = compute_gravity(Xobs, Yobs, Zobs, xe, ye, ze, build_rho(recovered))
residual = gz_rec - gz_obs
rms_grav  = float(np.sqrt(np.mean(residual**2)))
rms_depth = float(np.sqrt(np.mean((recovered - basin_true)**2)))

print(f'RMS gravity residual : {rms_grav:.4f} mGal')
print(f'RMS depth error      : {rms_depth:.1f} m')
print(f'True max depth       : {basin_true.max():.1f} m')
print(f'Recovered max depth  : {recovered.max():.1f} m')

## Plot (a): Gravity anomaly maps — observed, recovered, residual

This plotting cell generates maps of the observed gravity anomaly, recovered anomaly, and residuals over the twin basin.
Consistent colour limits and labels are applied, the figure is saved to disk, and the plots help visually assess how well the inversion reproduces the true gravity signal.

In [ ]:
# ── Plot (a): Gravity anomaly maps ──────────────────────────────────────────
xkm = xc / 1e3;  ykm = yc / 1e3
XX, YY = np.meshgrid(xkm, ykm, indexing='ij')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('(a)  Gravity Anomaly Maps — Model 2 (Twin Basin, Constant Density)',
             fontsize=13, fontweight='bold')

gzlo = float(min(gz_obs.min(), gz_rec.min()))
gzhi = float(max(gz_obs.max(), gz_rec.max()))
vr   = float(np.max(np.abs([residual.min(), residual.max()])))

for ax, dat, ttl, cmp, (vlo, vhi) in zip(
        axes,
        [gz_obs,   gz_rec,   residual],
        ['Observed Anomaly (mGal)', 'Recovered Anomaly (mGal)', 'Residual Anomaly (mGal)'],
        ['jet',    'jet',    'RdBu_r'],
        [(gzlo, gzhi), (gzlo, gzhi), (-vr, vr)]):
    cf = ax.contourf(XX, YY, dat,
                     levels=np.linspace(vlo, vhi, 200), cmap=cmp, extend='both')
    ax.axhline(7.5, color='white', linestyle='--', linewidth=1.5)
    ax.set_title(ttl, fontweight='bold', fontsize=11)
    ax.set_xlabel('x (km)', fontweight='bold', fontsize=11)
    ax.set_ylabel('y (km)', fontweight='bold', fontsize=11)
    ax.set_xlim(0, 15);  ax.set_ylim(0, 15)
    ax.xaxis.set_major_locator(MultipleLocator(5))
    ax.yaxis.set_major_locator(MultipleLocator(5))
    ax.grid(True, linestyle='--', linewidth=0.4, alpha=0.5, color='k')
    cb = plt.colorbar(cf, ax=ax, pad=0.02)
    cb.locator   = mticker.MaxNLocator(nbins=6)
    cb.formatter = mticker.FormatStrFormatter('%.0f')
    cb.update_ticks()
    cb.set_label('mGal', fontsize=10)

plt.tight_layout()
plt.savefig('model2_a_gravity.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: model2_a_gravity.png')

## Plot (b): Depth maps — true vs recovered twin-basin geometry

Here true and recovered basin depth maps are plotted side by side, often with a difference or error map.[file:3]  
The visual comparison reveals where geometry is well recovered and where the inversion overestimates or underestimates depth, especially around basin edges and overlap regions.[file:3]

In [ ]:
# ── Plot (b): Basin depth maps ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle('(b)  Basin Depth Maps — Model 2 (Twin Basin, Constant Density)',
             fontsize=13, fontweight='bold')

vmax_d = float(max(basin_true.max(), recovered.max()))
lvs_d  = np.linspace(0, vmax_d, 60)

last_cf = None
for ax, dat, ttl in zip(
        axes,
        [basin_true, recovered],
        ['True Basin Depth (m)', 'Recovered Basin Depth (m)']):
    cf = ax.contourf(XX, YY, dat, levels=lvs_d, cmap='jet', extend='max')
    ax.axhline(7.5, color='white', linestyle='--', linewidth=2.0)
    ax.set_title(ttl, fontweight='bold', fontsize=12)
    ax.set_xlabel('x (km)', fontweight='bold', fontsize=11)
    ax.set_ylabel('y (km)', fontweight='bold', fontsize=11)
    ax.set_xlim(0, 15);  ax.set_ylim(0, 15)
    ax.xaxis.set_major_locator(MultipleLocator(5))
    ax.yaxis.set_major_locator(MultipleLocator(5))
    ax.grid(True, linestyle='--', linewidth=0.4, alpha=0.5, color='k')
    last_cf = cf

fig.subplots_adjust(bottom=0.20)
cax = fig.add_axes([0.15, 0.06, 0.70, 0.03])
cb  = fig.colorbar(last_cf, cax=cax, orientation='horizontal')
cb.set_label('Depth (m)', fontweight='bold', fontsize=12)

plt.savefig('model2_b_depth.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: model2_b_depth.png')

## Plot (c): Cross-section and over/under-estimation shading

This cell extracts a representative cross-section through both basins and plots true vs recovered profiles along that line.
Additional shading indicates zones where the recovered surface lies above or below the true surface, providing an intuitive view of spatially systematic depth errors.

In [ ]:
# ── Plot (c): Vertical cross-section at Y = 7.5 km ─────────────────────────
j = int(np.argmin(np.abs(yc - 7500.0)))
print(f'Cross-section at yc[{j}] = {yc[j]/1e3:.3f} km  (target 7.5 km)')

true_slice = basin_true[:, j]
rec_slice  = recovered[:, j]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(xkm, true_slice,
        linestyle='-', marker='o', color='blue', linewidth=2.2, label='True Basin')
ax.plot(xkm, rec_slice,
        color='red', linewidth=2.7, label='Recovered Basin')
ax.fill_between(xkm, true_slice, rec_slice,
                where=(rec_slice > true_slice),
                alpha=0.25, color='orange', label='Over-estimated')
ax.fill_between(xkm, true_slice, rec_slice,
                where=(rec_slice < true_slice),
                alpha=0.25, color='green',  label='Under-estimated')
ax.axvline(x1 / 1e3, color='blue',  linestyle=':', linewidth=1.2, alpha=0.7,
           label=f'Basin 1 ({x1/1e3:.1f} km)')
ax.axvline(x2 / 1e3, color='green', linestyle=':', linewidth=1.2, alpha=0.7,
           label=f'Basin 2 ({x2/1e3:.1f} km)')

ax.set_title('(c)  Vertical Cross-Section at Y = 7.5 km',
             fontweight='bold', fontsize=14)
ax.set_xlabel('x (km)', fontweight='bold', fontsize=13)
ax.set_ylabel('Depth (m)', fontweight='bold', fontsize=13)
ax.invert_yaxis();  ax.set_xlim(0, 15)
ax.xaxis.set_major_locator(MultipleLocator(2.5))
ax.yaxis.set_major_locator(MultipleLocator(400))
for l in ax.get_xticklabels() + ax.get_yticklabels():
    l.set_fontweight('bold')
ax.tick_params(labelsize=12)
ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
ax.legend(prop={'weight': 'bold', 'size': 10}, loc='lower center', ncol=3)
ax.text(0.02, 0.05,
        f'RMS gravity: {rms_grav:.3f} mGal\nRMS depth: {rms_depth:.1f} m',
        transform=ax.transAxes, fontsize=10, verticalalignment='bottom',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.75))

plt.tight_layout()
plt.savefig('model2_c_xsection.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: model2_c_xsection.png')

## Final Summary

In [ ]:
# ── Final summary ────────────────────────────────────────────────────────────
print('=' * 70)
print('  FINAL SUMMARY — Model 2  (Twin Basin, Constant Density)')
print('=' * 70)
print(f'  Grid            : {nx}x{ny}x{nz}  vertical extent: {max_depth:.0f} m')
print(f'  Control pts     : {n_cx}x{n_cy} = {n_cx*n_cy}  (was 12x12=144)')
print(f'  Depth bounds    : [{dlo:.0f}, {dhi:.0f}] m')
print(f'  Regularisation  : lambda_s={lambda_s}  lambda_d={lambda_d}')
print(f'  DE  strategy=randtobest1bin  converged={de.success}  misfit={de.fun:.8f}')
print(f'  LB  converged={lb.success}   misfit={lb.fun:.8f}')
print(f'  LB2 converged={lb2_result.success}  misfit={lb2_result.fun:.8f}')
#print(f'  LB3 converged={lb3.success}  misfit={lb3.fun:.8f}')
print(f'  Best data misfit : {best_f:.8f}')
print(f'  RMS gravity      : {rms_grav:.4f} mGal')
print(f'  RMS depth        : {rms_depth:.1f} m')
print(f'  True max depth   : {basin_true.max():.1f} m')
print(f'  Rec  max depth   : {recovered.max():.1f} m')
print('=' * 70)


## DE Convergence Curve

In [ ]:
# ── Plot (d): DE convergence curve ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogy(range(1, len(_hist_f) + 1), _hist_f,
            '-o', markersize=3, color='steelblue', linewidth=1.8)
ax.set_title('DE Convergence Curve — Model 2 (Twin Basin - Constant Density)',
             fontweight='bold', fontsize=13)
ax.set_xlabel('DE Iteration', fontweight='bold', fontsize=12)
ax.set_ylabel('Best Misfit (normalised MSE)', fontweight='bold', fontsize=12)
ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
for l in ax.get_xticklabels() + ax.get_yticklabels():
    l.set_fontweight('bold')
ax.tick_params(labelsize=11)
plt.tight_layout()
plt.savefig('model2_de_convergence.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: model2_de_convergence.png')

## Cost-function topography in PCA space

In [ ]:
# ── 1. Build ensemble of "acceptable" models and their misfits ───────────────
misfit_array = np.array(_hist_f)
models_array = np.array(_hist_x)          # shape (N_models, n_params)
models_array = models_array.T             # shape (n_params, N_models)

misfit_threshold = np.percentile(misfit_array, 40.0)  # best 40% of models
mask_ok = misfit_array <= misfit_threshold

cost_finall  = misfit_array[mask_ok]     # (N_ok,)
model_finall = models_array[:, mask_ok]  # (n_params, N_ok)

print(f"Accepted models for PCA: {model_finall.shape[1]}")

# ── 2. PCA reduction ──────────────────────────────────────────────────────────
def pca_reduction_py(data):
    mean_vec = np.mean(data, axis=1, keepdims=True)
    data_z   = data - mean_vec
    C = np.cov(data_z)
    Evals, W_col = np.linalg.eigh(C)
    idx     = np.argsort(Evals)[::-1]
    Evalues = Evals[idx]
    W       = W_col[:, idx].T   # rows = eigenvectors
    pc      = W @ data_z
    return pc, Evalues, W, mean_vec

pc, Evalues, W, mean_model = pca_reduction_py(model_finall)

# ── 3. Cost-function topography in the PC1-PC2 plane ─────────────────────────
x = pc[0, :]   # PC1 scores
y = pc[1, :]   # PC2 scores

nxg, nyg = 80, 80
xg = np.linspace(x.min(), x.max(), nxg)
yg = np.linspace(y.min(), y.max(), nyg)
Xg, Yg = np.meshgrid(xg, yg, indexing="ij")

Vq = griddata(points=np.vstack([x, y]).T,
              values=cost_finall,
              xi=(Xg, Yg),
              method="linear")

plt.figure(figsize=(7, 5))
cs   = plt.contourf(Xg, Yg, Vq, levels=12, cmap="jet")
cbar = plt.colorbar(cs)
cbar.set_label("Regularised misfit (dimensionless)")
plt.xlabel("Principal component 1")
plt.ylabel("Principal component 2")
plt.title("Cost-function topography in PCA space (twin basin-constant density) noisy data")

# ── 4. Project best model and true model into PCA space ──────────────────────
# FIX: Use best_x (post all polishing), and Z_basin_true sampled on ctrl grid.
params_best = best_x

from scipy.interpolate import RegularGridInterpolator
_interp_true = RegularGridInterpolator(
    (xc, yc), basin_true, method="linear",
    bounds_error=False, fill_value=0.0)
_X_ctrl_2d, _Y_ctrl_2d = np.meshgrid(x_ctrl, y_ctrl, indexing="ij")
true_on_ctrl = _interp_true(
    np.column_stack([_X_ctrl_2d.ravel(), _Y_ctrl_2d.ravel()])
).reshape(n_cx, n_cy)
true_model = true_on_ctrl.ravel()

mean_flat     = mean_model.ravel()
best_centered = params_best - mean_flat
true_centered = true_model  - mean_flat

loc_best = W @ best_centered
loc_true = W @ true_centered

plt.plot(loc_best[0], loc_best[1], "r^", markersize=10, label="Best model (post L-BFGS-B)")
plt.plot(loc_true[0], loc_true[1], "gv", markersize=10, label="True model")
plt.legend(loc="best")
plt.tight_layout()
plt.savefig('model2_pca_noisy.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Best model  PC1={loc_best[0]:.4f}  PC2={loc_best[1]:.4f}")
print(f"True model  PC1={loc_true[0]:.4f}  PC2={loc_true[1]:.4f}")
dist = np.sqrt((loc_best[0]-loc_true[0])**2 + (loc_best[1]-loc_true[1])**2)
print(f"PC-space distance: {dist:.4f}")
